# CNN-LSTM Baseline + Naive — NCKH Table 1 (GH-69)

Train CNN-LSTM baseline (kiến trúc chuẩn trong `.claude/rules/tech/ai.md`) với
**ĐÚNG protocol bài báo**: split CŨ 23/2/1 theo battery ID (B0047 ở VAL —
notebook tự vá `preprocess.py` vì dev đã đổi sang 24/1/1 từ GH-88),
seed 42, cùng 6 features, window=30. Kèm naive baseline (SOH cycle t = SOH cycle t−1).

Output: MAE/RMSE trên B0048 → điền 2 dòng còn trống của **Table 1**.

**Setup:** Accelerator = **GPU T4 x2** (KHÔNG chọn P100 — Kaggle PyTorch đã bỏ hỗ trợ sm_60)
· Internet On · Add Data `nasa-battery-dataset` · Secret `GITHUB_TOKEN`.
Chạy: Save Version → Save & Run All. Ước lượng ~30–60 phút.

## 1. GPU + clone repo (branch `dev`)

In [ ]:
import subprocess

import torch
print("CUDA:", torch.cuda.is_available())

REPO_URL = "github.com/GSU26SE55/ai-module.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@{REPO_URL}"
except Exception:
    clone_url = f"https://{REPO_URL}"

subprocess.run(["git", "clone", "--branch", "dev", "--single-branch",
                clone_url, "/kaggle/working/ai-module"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/ai-module", "remote", "set-url",
                "origin", f"https://{REPO_URL}"], check=True)
commit = subprocess.run(["git", "-C", "/kaggle/working/ai-module", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Commit:", commit)

## 2. Deps + ép split CŨ 23/2/1 + preprocess window-30

⚠️ GH-88 (2026-07-08) đã chuyển B0047 vào TRAIN trên `dev` (split 24/1/1).
Bài báo dùng split cũ (B0047 ở VAL) — baseline PHẢI cùng protocol với
Mamba long v2.2 / window-30 v1.5 (đều train trước GH-88), nếu không Table 1 vô nghĩa.

In [ ]:
%pip install -q scipy scikit-learn

import os

REPO = "/kaggle/working/ai-module"
DATASET = "/kaggle/input/nasa-battery-dataset/cleaned_dataset"
PROCESSED = "/kaggle/working/processed"

if not os.path.isfile(f"{DATASET}/metadata.csv"):
    import glob
    hits = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
    assert hits, "Khong tim thay metadata.csv — kiem tra Add Data"
    DATASET = os.path.dirname(hits[0])

os.makedirs(PROCESSED, exist_ok=True)
os.chdir(REPO)
print("DATASET =", DATASET)

# ===== ep split CU 23/2/1: bo B0047 khoi TRAIN, tra ve VAL =====
prep = f"{REPO}/scripts/preprocess.py"
src = open(prep, encoding="utf-8").read()
b0047_line = '    "B0047",  # 4\u00b0C, SOH 0-83.7% \u2014 GH-88: fills the missing 4\u00b0C high-SOH region\n'
assert b0047_line in src and 'VAL_IDS = ["B0046"]' in src, "preprocess.py da doi — cap nhat patch"
src = src.replace(b0047_line, "")
src = src.replace('VAL_IDS = ["B0046"]', 'VAL_IDS = ["B0046", "B0047"]')
open(prep, "w", encoding="utf-8").write(src)
print("Da ep split cu: 23 train / 2 val (B0046,B0047) / 1 test (B0048)")

!python scripts/preprocess.py --data-dir "{DATASET}" --output-dir "{PROCESSED}"

## 3. Load data

In [ ]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

splits = {}
for name in ["train", "val", "test"]:
    d = torch.load(f"{PROCESSED}/{name}.pt", weights_only=False)
    splits[name] = d
    print(f"{name}: X={tuple(d['X'].shape)}, y={tuple(d['y'].shape)}")

n_features = splits["train"]["X"].shape[-1]
print("n_features =", n_features)

def to_tensor(d):
    X = torch.as_tensor(np.asarray(d["X"]), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(d["y"]), dtype=torch.float32)
    return X, y

Xtr, ytr = to_tensor(splits["train"])
Xva, yva = to_tensor(splits["val"])
Xte, yte = to_tensor(splits["test"])

## 4. CNN-LSTM baseline — kiến trúc chuẩn từ rules

`Conv1d(F→32,k3) → ReLU → MaxPool(2) → LSTM(32→64,×2,dropout 0.2) → FC 64→32→1`.
Training config theo rules: Adam lr=1e-3, MSELoss, 50 epochs, patience=10, batch=32.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class SOHPredictor(nn.Module):
    """CNN-LSTM baseline — in_channels theo so feature thuc te (6)."""
    def __init__(self, in_features):
        super().__init__()
        self.conv1 = nn.Conv1d(in_features, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(32, 64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):                    # x: (B, 30, F)
        x = x.permute(0, 2, 1)               # (B, F, 30)
        x = self.pool(self.relu(self.conv1(x)))   # (B, 32, 15)
        x = x.permute(0, 2, 1)               # (B, 15, 32)
        _, (h_n, _) = self.lstm(x)
        x = self.dropout(self.relu(self.fc1(h_n[-1])))
        return self.fc2(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
model = SOHPredictor(n_features).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"CNN-LSTM params: {n_params:,}")

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()
gen = torch.Generator(); gen.manual_seed(SEED)
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True, generator=gen)

def evaluate(X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X.to(device)).cpu() * 100.0
    mae = float(torch.mean(torch.abs(pred - y)))
    rmse = float(torch.sqrt(torch.mean((pred - y) ** 2)))
    return mae, rmse

best_val, best_state, patience_ctr = float("inf"), None, 0
for epoch in range(1, 51):
    model.train()
    for Xb, yb in loader:
        opt.zero_grad(set_to_none=True)
        loss = crit(model(Xb.to(device)), (yb / 100.0).to(device))
        loss.backward()
        opt.step()
    val_mae, _ = evaluate(Xva, yva)
    marker = ""
    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0; marker = " *"
    else:
        patience_ctr += 1
    print(f"epoch {epoch:02d}  val MAE {val_mae:.4f}%{marker}")
    if patience_ctr >= 10:
        print("early stop"); break

model.load_state_dict(best_state)
test_mae, test_rmse = evaluate(Xte, yte)
print("=" * 50)
print(f"CNN-LSTM  B0048: MAE {test_mae:.4f}%  RMSE {test_rmse:.4f}%  (params {n_params:,})")

## 5. Naive baseline — SOH(t) = SOH(t−1) ở mức cycle trên B0048

In [ ]:
import sys
sys.path.insert(0, REPO)
from scripts.preprocess import load_cycles

# load_cycles tra (cycle_array, soh, cycle_idx) — lay phan tu thu 2 (soh)
cycles = load_cycles(DATASET, "B0048")
soh_series = np.array([c[1] for c in cycles], dtype=float)
pred, true = soh_series[:-1], soh_series[1:]
naive_mae = float(np.mean(np.abs(pred - true)))
naive_rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
print(f"Naive last-SOH  B0048 ({len(true)} cycles): MAE {naive_mae:.4f}%  RMSE {naive_rmse:.4f}%")

## 6. Lưu kết quả (tab Output)

In [ ]:
import json

results = {
    "commit": commit,
    "protocol": "OLD split 23/2/1 by battery ID (B0047 in VAL — paper protocol), seed 42, window=30, 6 features",
    "cnn_lstm": {"params": n_params, "test_mae_pct": round(test_mae, 4),
                  "test_rmse_pct": round(test_rmse, 4), "best_val_mae_pct": round(best_val, 4)},
    "naive_last_soh": {"test_mae_pct": round(naive_mae, 4), "test_rmse_pct": round(naive_rmse, 4)},
}
os.makedirs("/kaggle/working/baseline_results", exist_ok=True)
with open("/kaggle/working/baseline_results/table1_baselines.json", "w") as f:
    json.dump(results, f, indent=2)
torch.save({"model_state_dict": best_state, "params": n_params, "seed": SEED},
           "/kaggle/working/baseline_results/cnn_lstm_baseline.pth")
print(json.dumps(results, indent=2))
print("\nTai ve tu tab Output -> dat vao logs/nckh/baseline/ va commit.")